In [ ]:
from pathlib import Path

import numpy as np
import seaborn as sns
import xarray as xr
from bonner.plotting import save_figure
from matplotlib import pyplot as plt
from tqdm.auto import tqdm

from lib.datasets import (
    compute_shared_stimuli,
    filter_by_stimulus,
    nsd,
    split_by_repetition,
)
from lib.spectra import (
    bin_data,
    compute_spectra_with_n_fold_cross_validation,
    extract_geometrically_spaced_bins,
    plot_spectra,
)
from lib.utilities import JOURNAL_MATPLOTLIBRC, mathtext_exponent_label

FIGURES_HOME = Path.cwd().parent / "figures"
FIGURES_HOME.mkdir(exist_ok=True, parents=True)

sns.set_theme(context="paper", style="ticks", rc=JOURNAL_MATPLOTLIBRC)

REFERENCE_SUBJECT = 0

In [ ]:
dataset = nsd.load_dataset(
    subject=REFERENCE_SUBJECT,
    preprocessing="fithrf",
    roi="general",
    z_score=True,
)
shared_stimuli = compute_shared_stimuli([dataset], n_repetitions=2)

rng = np.random.default_rng(seed=0)

In [ ]:
n_stimuli = np.geomspace(766, 1e4, num=5).astype(int)

bin_edges, bin_centers = extract_geometrically_spaced_bins(
    start=1,
    stop=10_000,
    density=3,
)

spectra = []
for n_stimuli_ in tqdm(n_stimuli, desc="n_stimuli", leave=False):
    if n_stimuli_ == 10_000:
        datasets = split_by_repetition(
            filter_by_stimulus(
                dataset,
                stimuli=shared_stimuli,
            ),
            n_repetitions=2,
        )
    else:
        datasets = split_by_repetition(
            filter_by_stimulus(
                dataset,
                stimuli=set(rng.permutation(sorted(shared_stimuli))[:n_stimuli_]),
            ),
            n_repetitions=2,
        )

    spectra_ = compute_spectra_with_n_fold_cross_validation(
        x_train=datasets[0],
        y_train=datasets[1],
        x_test=datasets[0],
        y_test=datasets[1],
        n_folds=8,
        n_permutations=5_000,
    )
    spectra_ = bin_data(
        spectra_,
        bin_edges={"component": bin_edges},
        bin_centers={"component": bin_centers},
        dim="rank",
    ).expand_dims(n_stimuli=[n_stimuli_])
    spectra.append(spectra_)

spectra = xr.concat(spectra, dim="n_stimuli")

In [ ]:
fig, ax = plt.subplots(figsize=(3, 3))

kwargs_legend = {
    "loc": "upper right",
    "title": "number of stimuli",
    "ncols": 1,
    "columnspacing": 0.5,
    "handletextpad": 0.25,
    "reverse": True,
}

plot_spectra(
    ax=ax,
    spectra=spectra,
    hue="n_stimuli",
    palette="inferno_r",
    hue_order=list(reversed(n_stimuli)),
    hue_labels=[f"{n_train}" for n_train in reversed(n_stimuli)],
    marker="s",
    hide_insignificant=True,
    null_quantile=0.999,
)
ax.set_title("within-subject, subject 1", pad=10)
ax.set_ylabel("covariance")
ax.set_xlabel("rank")
ax.legend(**kwargs_legend)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(left=1, right=1e4)
ax.set_xticks([1, 1e1, 1e2, 1e3, 1e4])
ax.set_ylim(top=1e-1, bottom=1e-8)
ytick_exponents = list(range(-8, 0))
ax.set_yticks(
    [10**exponent for exponent in ytick_exponents],
    labels=[
        mathtext_exponent_label(exponent) if exponent % 2 == 1 else ""
        for exponent in ytick_exponents
    ],
)

save_figure(fig, filepath=FIGURES_HOME / "vary-n-stimuli.pdf")